In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import gc
import random
import math
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
import copy

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

def clear_variable(var_list):
    for var in var_list:
        globals().pop(var, None)
    gc.collect()

file_list = [f"Data_unique/3-mer/rnn_data_subset_{i+1}.csv" for i in range(20)]
random.shuffle(file_list)

num_folds = 5
fold_size = len(file_list) // num_folds
folds = [file_list[i * fold_size : (i + 1) * fold_size] for i in range(num_folds)]


class CustomDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class CNNLSTM(nn.Module):
    def __init__(
        self,
        input_length,
        vocab_size=65,
        embedding_dim=16,
        num_classes=8,
    ):
        super(CNNLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.conv1 = nn.Conv1d(embedding_dim, 32, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool1d(kernel_size=4)
        self.drop1 = nn.Dropout(0.2)

        self.conv2 = nn.Conv1d(32, 64, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool1d(kernel_size=4)
        self.drop2 = nn.Dropout(0.2)

        self.conv3 = nn.Conv1d(64, 128, kernel_size=5, padding=2)
        self.pool3 = nn.MaxPool1d(kernel_size=4)
        self.drop3 = nn.Dropout(0.2)

        self.conv4 = nn.Conv1d(128, 256, kernel_size=5, padding=2)
        self.pool4 = nn.MaxPool1d(kernel_size=4)
        self.drop4 = nn.Dropout(0.2)

        self.lstm1 = nn.LSTM(256, 64, batch_first=True, bidirectional=True)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop_lstm1 = nn.Dropout(0.2)

        self.lstm2 = nn.LSTM(128, 128, batch_first=True, bidirectional=True)
        self.bn2 = nn.BatchNorm1d(256)
        self.drop_lstm2 = nn.Dropout(0.2)

        self.fc1 = nn.Linear(256, 256)
        self.drop_fc1 = nn.Dropout(0.3)
        self.fc_out = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.extract_features(x)
        x = torch.relu(self.fc1(x))
        x = self.drop_fc1(x)
        return self.fc_out(x)

    def extract_features(self, x):
        x = self.embedding(x).transpose(1, 2)

        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)
        x = self.drop1(x)

        x = torch.tanh(self.conv2(x))
        x = self.pool2(x)
        x = self.drop2(x)

        x = torch.tanh(self.conv3(x))
        x = self.pool3(x)
        x = self.drop3(x)

        x = torch.tanh(self.conv4(x))
        x = self.pool4(x)
        x = self.drop4(x)

        x = x.transpose(1, 2)
        x, _ = self.lstm1(x)
        x = self.bn1(x.transpose(1, 2)).transpose(1, 2)
        x = self.drop_lstm1(x)

        x, _ = self.lstm2(x)
        x = x[:, -1, :]
        x = self.bn2(x)
        return self.drop_lstm2(x)


def init_weights(m):
    if isinstance(m, nn.Conv1d):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.LSTM):
        for name, param in m.named_parameters():
            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)
            elif "weight_hh" in name:
                nn.init.orthogonal_(param)
            elif "bias" in name:
                nn.init.zeros_(param)
    elif isinstance(m, nn.Embedding):
        nn.init.uniform_(m.weight, -0.05, 0.05)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def create_model(input_length, vocab_size=65, embedding_dim=16, num_classes=8):
    model = CNNLSTM(input_length, vocab_size, embedding_dim, num_classes)
    model.apply(init_weights)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    return model, optimizer, criterion


def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    return running_loss / total, correct / total


def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            probs = torch.softmax(outputs, dim=1)
            all_probs.extend(probs.cpu().numpy())
            preds = outputs.argmax(dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
    return (
        running_loss / total,
        correct / total,
        np.array(all_labels),
        np.array(all_preds),
        np.array(all_probs),
    )


def get_features_from_loader(model, dataloader):
    model.eval()
    features_list, labels_list = [], []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            feats = model.extract_features(inputs)
            features_list.append(feats.cpu().numpy())
            labels_list.append(labels.cpu().numpy())
    return np.concatenate(features_list, axis=0), np.concatenate(labels_list, axis=0)


class_names = [
    "B.1.1.7",
    "BA.1.1",
    "BA.2.12.1",
    "BA.4.6",
    "BA.5.4",
    "BQ.1.1",
    "B.1.617.2",
    "P.1",
]

tab10_8 = ListedColormap(plt.cm.tab10.colors[:8])

fold_results = []
sequence_length = 30000
num_epochs = 100
patience = 10

for fold_idx in range(num_folds):
    print(f"\nProcessing fold {fold_idx+1}/{num_folds}")
    test_files = folds[fold_idx]
    train_files = [
        f for j, f_list in enumerate(folds) if j != fold_idx for f in f_list
    ]

    X_train_list, X_val_list = [], []
    y_train_list, y_val_list = [], []
    for file in train_files:
        df = pd.read_csv(file)
        data_features = df.drop("class", axis=1).values
        data_labels = df["class"].values
        clear_variable(["df"])
        num_samples = data_features.shape[0]
        indices = np.random.permutation(num_samples)
        train_split = int(num_samples * 0.85)
        train_idx, val_idx = indices[:train_split], indices[train_split:]
        X_train_list.append(data_features[train_idx])
        X_val_list.append(data_features[val_idx])
        y_train_list.append(data_labels[train_idx])
        y_val_list.append(data_labels[val_idx])
        print("Train/Val data appended to the list")
    print("Train/Val data fully appended ✅")

    X_train = np.concatenate(X_train_list, axis=0)
    clear_variable(["X_train_list"])
    print("X_train data is ready ✅")

    X_val = np.concatenate(X_val_list, axis=0)
    clear_variable(["X_val_list"])
    print("X_val data is ready ✅")

    y_train = np.concatenate(y_train_list, axis=0)
    clear_variable(["y_train_list"])
    print("y_train data is ready ✅")

    y_val = np.concatenate(y_val_list, axis=0)
    clear_variable(["y_val_list"])
    print("y_val data is ready ✅")

    X_test_list, y_test_list = [], []
    for file in test_files:
        df = pd.read_csv(file)
        data_features = df.drop("class", axis=1).values
        data_labels = df["class"].values
        clear_variable(["df"])
        X_test_list.append(data_features)
        y_test_list.append(data_labels)
        print("Test data appended to the list")
    print("Test data fully appended ✅")

    X_test = np.concatenate(X_test_list, axis=0)
    clear_variable(["X_test_list"])
    print("X_test data is ready ✅")

    y_test = np.concatenate(y_test_list, axis=0)
    clear_variable(["y_test_list"])
    print("y_test data is ready ✅")

    y_train -= 1
    y_val -= 1
    y_test -= 1

    print(
        f"Fold {fold_idx+1} shapes -- "
        f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}"
    )

    tsne = TSNE(n_components=2, random_state=seed)
    X_train_tsne = tsne.fit_transform(X_train)

    plt.figure(figsize=(8, 6), dpi=300)
    scatter = plt.scatter(
        X_train_tsne[:, 0],
        X_train_tsne[:, 1],
        c=y_train,
        cmap=tab10_8,
        alpha=0.7,
    )
    plt.title(f"Fold {fold_idx+1} - t-SNE of Raw Training Data")
    plt.xlabel("t-SNE Feature 1")
    plt.ylabel("t-SNE Feature 2")
    cbar = plt.colorbar(scatter, ticks=range(8))
    cbar.ax.set_yticklabels(class_names)
    plt.show()

    train_dataset = CustomDataset(X_train, y_train)
    val_dataset = CustomDataset(X_val, y_val)
    test_dataset = CustomDataset(X_test, y_test)

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    model, optimizer, criterion = create_model(sequence_length)

    best_val_loss = float("inf")
    epochs_no_improve = 0
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_model_state = None
    lr_current = 0.001

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, criterion
        )
        val_loss, val_acc, _, _, _ = evaluate(model, val_loader, criterion)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(
            f"Epoch {epoch+1}/{num_epochs} -- "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

        if epoch >= 5:
            lr_current *= math.exp(-0.1)
            for param_group in optimizer.param_groups:
                param_group["lr"] = lr_current

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            best_model_state = copy.deepcopy(model.state_dict())
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered at epoch {epoch+1}")
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    test_loss, test_acc, y_true, y_pred, y_pred_prob = evaluate(
        model, test_loader, criterion
    )
    print(f"Fold {fold_idx+1} Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

    train_loader_no_shuffle = DataLoader(
        train_dataset, batch_size=32, shuffle=False
    )
    features_train, labels_train = get_features_from_loader(
        model, train_loader_no_shuffle
    )

    tsne2 = TSNE(n_components=2, random_state=seed)
    features_train_tsne = tsne2.fit_transform(features_train)

    plt.figure(figsize=(8, 6), dpi=300)
    scatter2 = plt.scatter(
        features_train_tsne[:, 0],
        features_train_tsne[:, 1],
        c=labels_train,
        cmap=tab10_8,
        alpha=0.7,
    )
    plt.title(f"Fold {fold_idx+1} - t-SNE of Learned Features (Training Data)")
    plt.xlabel("t-SNE Feature 1")
    plt.ylabel("t-SNE Feature 2")
    cbar2 = plt.colorbar(scatter2, ticks=range(8))
    cbar2.ax.set_yticklabels(class_names)
    plt.show()

    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Fold {fold_idx+1} Accuracy Curve")
    plt.legend()
    plt.show()

    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"Fold {fold_idx+1} Zoomed-in Accuracy Curve")
    plt.ylim(0.94, 1.0)
    plt.legend()
    plt.show()

    n_classes = len(class_names)
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

    fpr, tpr, roc_auc, prec, rec, avg_prec = {}, {}, {}, {}, {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_pred_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
        prec[i], rec[i], _ = precision_recall_curve(
            y_true_bin[:, i], y_pred_prob[:, i]
        )
        avg_prec[i] = average_precision_score(
            y_true_bin[:, i], y_pred_prob[:, i]
        )

    fpr["micro"], tpr["micro"], _ = roc_curve(
        y_true_bin.ravel(), y_pred_prob.ravel()
    )
    roc_auc["micro"] = auc(fpr["micro"], tpr["micro"])
    prec["micro"], rec["micro"], _ = precision_recall_curve(
        y_true_bin.ravel(), y_pred_prob.ravel()
    )
    avg_prec["micro"] = average_precision_score(
        y_true_bin, y_pred_prob, average="micro"
    )

    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(
        fpr["micro"],
        tpr["micro"],
        label=f"Micro-average ROC (AUC = {roc_auc['micro']:.2f})",
        lw=2,
    )
    for i in range(n_classes):
        plt.plot(
            fpr[i],
            tpr[i],
            label=f"Class {class_names[i]} (AUC = {roc_auc[i]:.2f})",
            lw=1.5,
        )
    plt.plot([0, 1], [0, 1], "k--", lw=2)
    plt.xlim(-0.02, 1.02)
    plt.ylim(-0.02, 1.02)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Fold {fold_idx+1} - Multi-Class ROC Curves")
    plt.legend(loc="lower right")
    plt.show()

    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(
        fpr["micro"],
        tpr["micro"],
        label=f"Micro-average ROC (AUC = {roc_auc['micro']:.2f})",
        lw=2,
    )
    for i in range(n_classes):
        plt.plot(
            fpr[i],
            tpr[i],
            label=f"Class {class_names[i]} (AUC = {roc_auc[i]:.2f})",
            lw=1.5,
        )
    plt.plot([0, 1], [0, 1], "k--", lw=2)
    plt.xlim(-0.005, 0.05)
    plt.ylim(0.96, 1.005)
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"Fold {fold_idx+1} - Zoomed-in ROC Curves")
    plt.legend(loc="lower right")
    plt.show()

    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(
        rec["micro"],
        prec["micro"],
        label=f"Micro-average PR (AP = {avg_prec['micro']:.2f})",
        lw=2,
    )
    for i in range(n_classes):
        plt.plot(
            rec[i],
            prec[i],
            label=f"Class {class_names[i]} (AP = {avg_prec[i]:.2f})",
            lw=1.5,
        )
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Fold {fold_idx+1} - Multi-Class Precision-Recall Curves")
    plt.legend(loc="lower left")
    plt.show()

    plt.figure(figsize=(8, 6), dpi=300)
    plt.plot(
        rec["micro"],
        prec["micro"],
        label=f"Micro-average PR (AP = {avg_prec['micro']:.2f})",
        lw=2,
    )
    for i in range(n_classes):
        plt.plot(
            rec[i],
            prec[i],
            label=f"Class {class_names[i]} (AP = {avg_prec[i]:.2f})",
            lw=1.5,
        )
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Fold {fold_idx+1} - Zoomed-in Precision-Recall Curves")
    plt.xlim(0.9, 1.01)
    plt.ylim(0.9, 1.01)
    plt.legend(loc="lower left")
    plt.show()

    fold_results.append(
        {
            "fold": fold_idx + 1,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
            "history": history,
            "y_test": y_true,
            "y_pred": y_pred,
            "y_pred_prob": y_pred_prob,
        }
    )

    clear_variable(
        ["X_train", "X_val", "y_train", "y_val", "X_test", "y_test"]
    )
    torch.cuda.empty_cache()

accuracies = [res["test_accuracy"] for res in fold_results]
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)
print(f"\nMean Test Accuracy: {mean_acc:.4f} +/- {std_acc:.4f}")

plt.figure(figsize=(6, 4), dpi=300)
plt.bar(1, mean_acc, yerr=std_acc, capsize=10)
plt.xlim(0, 2)
plt.xticks([1], ["Test Accuracy"])
plt.ylabel("Accuracy")
plt.title("Mean Test Accuracy with Error Bar")
plt.show()

all_y_true = np.concatenate([res["y_test"] for res in fold_results], axis=0)
all_y_pred = np.concatenate([res["y_pred"] for res in fold_results], axis=0)

agg_report = classification_report(all_y_true, all_y_pred, digits=4)
agg_conf_matrix = confusion_matrix(all_y_true, all_y_pred)

print("\nAggregate Classification Report:")
print(agg_report)
print("\nAggregate Confusion Matrix:")
print(agg_conf_matrix)

plt.figure(figsize=(8, 6), dpi=300)
sns.heatmap(agg_conf_matrix, annot=True, fmt="d", cmap=plt.cm.Blues)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Aggregate Confusion Matrix")
plt.show()